# RAG Übungen - Vollständige Lösungen

## **Lernziele**

Nach Abschluss dieser Übungen werden Sie:

- **Retrieval-Augmented Generation (RAG)** und seine Komponenten verstehen
- PDF-Dokumente effektiv laden, vorverarbeiten und verarbeiten können
- Textdaten in Embeddings für effizientes Retrieval umwandeln können
- Dokumenten-Retrieval-Systeme mit LangChain und FAISS implementieren und testen
- Retrieval-Systeme mit kostenlosen Language Models (LLMs) von ChatGroq integrieren
- Ein interaktives chatbasiertes Q&A-System aufbauen

---

## **Übung 1: Setup und Warm-up**

In dieser Übung richten Sie Ihre Umgebung ein und wählen ein geeignetes Sprachmodell aus.

**Schritte:**

1. **Umgebungsvariablen laden:** Stellen Sie sicher, dass Ihre Umgebungsvariablen (z.B. API-Schlüssel, Tokens) sicher gespeichert und geladen werden
2. **LLM auswählen:** Wählen Sie ein kostenloses LLM-Modell von ChatGroq
3. **Modell instanziieren:** Erstellen Sie eine Instanz Ihres gewählten Modells

### **Warum ist das wichtig?**
- **API-Schlüssel schützen:** Niemals Schlüssel direkt im Code speichern
- **Temperatur = 0:** Macht die Ausgabe deterministisch (gleiche Eingabe → gleiche Ausgabe)
- **max_retries:** Erhöht die Robustheit bei Netzwerkproblemen

In [1]:
# Notwendige Bibliotheken importieren
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import warnings

# Warnungen unterdrücken für saubere Ausgabe
warnings.filterwarnings("ignore")

# Umgebungsvariablen aus .env Datei laden
# Die .env Datei sollte enthalten: GROQ_API_KEY=your_api_key_here
load_dotenv()

print("✅ Umgebungsvariablen erfolgreich geladen!")

✅ Umgebungsvariablen erfolgreich geladen!


In [ ]:
# LLM Modell instanziieren
# Wir verwenden Llama 3.1 (8 Milliarden Parameter) - schnell und effizient
llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Modellname bei Groq
    temperature=0,                   # 0 = deterministisch, 1 = kreativ
    max_tokens=None,                 # Keine Begrenzung der Ausgabelänge
    timeout=None,                    # Keine Zeitbeschränkung
    max_retries=2                    # Bei Fehler 2x automatisch wiederholen
)

print("✅ LLM Modell erfolgreich initialisiert!")
print(f"📊 Modell: {llm.model}")

In [ ]:
# Test des LLM mit einer einfachen Frage
test_response = llm.invoke("Was ist Künstliche Intelligenz? Antworte in einem Satz.")
print("🤖 LLM Antwort:", test_response.content)

---

## **Übung 2: Datenaufnahme (Data Ingestion)**

In dieser Übung lernen Sie, wie man PDF-Daten in eine Python-Umgebung lädt.

**Schritte:**

1. **PDF Loader importieren:** Verwenden Sie LangChain's `PyPDFLoader`
2. **PDF-Datei laden:** Erstellen Sie eine Funktion zum Lesen der PDF-Datei
3. **PDF-Inhalt anzeigen:** Drucken Sie die Anzahl der Seiten und den Inhalt der ersten Seite

### **Warum ist das wichtig?**
- PDFs sind ein Standardformat für Dokumente
- PyPDFLoader extrahiert Text automatisch und bewahrt Metadaten (Seitenzahl, etc.)
- Jede Seite wird als separates "Document"-Objekt gespeichert

In [ ]:
# PyPDFLoader importieren
from langchain_community.document_loaders import PyPDFLoader

# Funktion zum Laden von PDF-Dateien
def load_pdf(pdf_path):
    """
    Lädt eine PDF-Datei und extrahiert den Text.
    
    Args:
        pdf_path (str): Pfad zur PDF-Datei
    
    Returns:
        list: Liste von Document-Objekten (jedes repräsentiert eine Seite)
    """
    # PDF-Loader erstellen
    loader = PyPDFLoader(file_path=pdf_path)
    
    # Dokumente laden (jede Seite = ein Document)
    documents = loader.load()
    
    print(f"✅ PDF erfolgreich geladen!")
    print(f"📄 Anzahl der Seiten: {len(documents)}")
    
    return documents

print("✅ PDF-Lade-Funktion definiert!")

In [ ]:
# Laden Sie das react_paper.pdf
react_documents = load_pdf(pdf_path="../documents/react_paper.pdf")

print("\n" + "="*80)
print("📖 ERSTE SEITE - VORSCHAU:")
print("="*80)
print(react_documents[0].page_content[:500])  # Erste 500 Zeichen
print("...")

print("\n" + "="*80)
print("📊 METADATEN:")
print("="*80)
print(react_documents[0].metadata)

---

## **Übung 3: Dokumenten-Chunking**

Diese Übung führt in das Aufteilen großer Dokumente in handhabbare Textblöcke ein.

**Schritte:**

1. **Text Splitter importieren:** Verwenden Sie `RecursiveCharacterTextSplitter`
2. **Dokument in Chunks aufteilen:** Schreiben Sie eine Funktion, die geladene Dokumente in Chunks aufteilt
3. **Funktion testen:** Überprüfen Sie durch Anzeige der resultierenden Chunks

### **Warum ist Chunking wichtig?**
- **LLMs haben begrenzte Kontextfenster:** Sie können nicht unbegrenzt Text verarbeiten
- **Bessere Retrieval-Qualität:** Kleinere Chunks = präzisere Suche
- **Chunk Overlap:** Verhindert, dass wichtige Informationen an Chunk-Grenzen verloren gehen

### **Parameter erklärt:**
- `chunk_size=200`: Jeder Chunk hat ca. 200 Zeichen
- `chunk_overlap=50`: 50 Zeichen überlappen zwischen Chunks
- Warum Overlap? Damit zusammenhängende Informationen nicht zerrissen werden

In [ ]:
# RecursiveCharacterTextSplitter importieren
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking-Funktion
def chunk_documents(documents, chunk_size=200, chunk_overlap=50):
    """
    Teilt Dokumente in kleinere Chunks auf.
    
    Args:
        documents (list): Liste von Document-Objekten
        chunk_size (int): Maximale Größe jedes Chunks in Zeichen
        chunk_overlap (int): Anzahl überlappender Zeichen zwischen Chunks
    
    Returns:
        list: Liste von Chunk-Dokumenten
    """
    # Text-Splitter initialisieren
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,  # Funktion zum Messen der Länge
        is_separator_regex=False  # Keine Regex für Trennzeichen
    )
    
    # Dokumente in Chunks aufteilen
    chunks = text_splitter.split_documents(documents=documents)
    
    # Jedem Chunk eine eindeutige ID zuweisen
    for i, chunk in enumerate(chunks):
        chunk.metadata.update({
            "chunk_id": f"chunk_{i}",
            "chunk_index": i
        })
    
    print(f"✅ Dokumente in Chunks aufgeteilt!")
    print(f"📦 Anzahl der Chunks: {len(chunks)}")
    print(f"📏 Chunk-Größe: {chunk_size} Zeichen")
    print(f"🔗 Chunk-Overlap: {chunk_overlap} Zeichen")
    
    return chunks

print("✅ Chunking-Funktion definiert!")

In [ ]:
# Führen Sie Ihre Chunking-Funktion aus
react_chunks = chunk_documents(react_documents, chunk_size=200, chunk_overlap=50)

print("\n" + "="*80)
print("📦 BEISPIEL-CHUNKS:")
print("="*80)

# Zeige die ersten 3 Chunks
for i in range(min(3, len(react_chunks))):
    print(f"\n--- CHUNK {i} ---")
    print(f"Text: {react_chunks[i].page_content[:150]}...")
    print(f"Metadaten: {react_chunks[i].metadata}")

---

## **Übung 4: Embedding und Speicherung**

In dieser Übung erstellen Sie Embeddings aus Text-Chunks und speichern sie effizient.

**Schritte:**

1. **Embedding-Modell auswählen:** Verwenden Sie `sentence-transformers/all-mpnet-base-v2` von Hugging Face
2. **Embeddings generieren:** Transformieren Sie Dokumenten-Chunks in Embeddings
3. **Embeddings speichern:** Speichern Sie diese Embeddings mit FAISS lokal

### **Was sind Embeddings?**
- **Numerische Darstellung von Text:** Jeder Text wird zu einem Vektor (Liste von Zahlen)
- **Semantische Bedeutung:** Ähnliche Texte haben ähnliche Vektoren
- **Dimensionen:** Das verwendete Modell erzeugt 768-dimensionale Vektoren

### **Warum Normalisierung?**
- **Cosine Similarity:** Vergleicht Richtung, nicht Länge der Vektoren
- **Ohne Normalisierung:** FAISS verwendet Dot Product (abhängig von Vektorlänge)
- **Mit Normalisierung:** Echte Cosine Similarity (nur Richtung zählt)

In [ ]:
# Bibliotheken importieren
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

# Funktion für Embeddings und Speicherung
def embed_and_store(chunks, db_name):
    """
    Erstellt Embeddings für Text-Chunks und speichert sie in einer FAISS-Vektordatenbank.
    
    Args:
        chunks (list): Liste von Dokumenten-Chunks
        db_name (str): Name der zu speichernden Datenbank
    
    Returns:
        FAISS: FAISS VectorStore-Objekt
    """
    print("⏳ Embedding-Modell wird initialisiert...")
    
    # Embedding-Modell instanziieren
    # Dieses Modell ist spezialisiert auf semantische Suche
    embedding_model = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-mpnet-base-v2',
        encode_kwargs={"normalize_embeddings": True}  # WICHTIG für Cosine Similarity!
    )
    
    print("⏳ Embeddings werden erstellt und gespeichert...")
    
    # FAISS VectorStore erstellen
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model,
        distance_strategy=DistanceStrategy.COSINE  # Cosine Similarity verwenden
    )
    
    # VectorStore lokal speichern
    save_path = f"../vector_databases/vector_db_{db_name}"
    vectorstore.save_local(save_path)
    
    print(f"✅ Embeddings erfolgreich erstellt und gespeichert!")
    print(f"💾 Speicherort: {save_path}")
    print(f"📊 Anzahl der Embeddings: {len(chunks)}")
    print(f"📏 Embedding-Dimensionen: 768 (vom Modell)")
    
    return vectorstore

print("✅ Embedding-Funktion definiert!")

In [ ]:
# Embeddings generieren und lokal speichern
react_vectorstore = embed_and_store(chunks=react_chunks, db_name="react")

print("\n" + "="*80)
print("🔍 VECTORSTORE-INFORMATIONEN:")
print("="*80)
print(f"Typ: {type(react_vectorstore)}")
print(f"Anzahl gespeicherter Dokumente: {react_vectorstore.index.ntotal}")

---

## **Übung 5: Retrieval aus FAISS**

Hier lernen Sie, wie man Dokumente aus einer Vektordatenbank mithilfe von Embeddings abruft.

**Schritte:**

1. **Embeddings laden:** Gespeicherte Embeddings aus der FAISS-Datenbank laden
2. **Retrieval implementieren:** Logik zum Abrufen relevanter Chunks basierend auf Anfragen erstellen
3. **Retriever testen:** Retrieval mit Beispielabfragen ausführen

### **Wie funktioniert Retrieval?**
1. **Anfrage wird zu Embedding:** Ihre Frage wird in einen Vektor umgewandelt
2. **Ähnlichkeitssuche:** FAISS findet die k ähnlichsten Vektoren in der Datenbank
3. **Dokumente zurückgeben:** Die zugehörigen Text-Chunks werden zurückgegeben

### **Wichtig:**
- **Gleiches Embedding-Modell verwenden:** Muss das gleiche sein wie beim Erstellen!
- **allow_dangerous_deserialization=True:** Nötig zum Laden gespeicherter FAISS-Datenbanken

In [ ]:
# Retrieval-Logik aus FAISS-Datenbank implementieren
def load_retriever_from_vectordb(vector_db_path):
    """
    Lädt eine FAISS-Vektordatenbank und erstellt ein Retriever-Objekt.
    
    Args:
        vector_db_path (str): Pfad zur gespeicherten Vektordatenbank
    
    Returns:
        tuple: (retriever, vectorstore)
    """
    print("⏳ Lade VectorStore...")
    
    # WICHTIG: Gleiches Embedding-Modell wie beim Erstellen verwenden!
    embedding_model = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-mpnet-base-v2',
        encode_kwargs={"normalize_embeddings": True}
    )
    
    # FAISS VectorStore laden
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding_model,
        allow_dangerous_deserialization=True,  # Nötig für pickle-Dateien
        distance_strategy=DistanceStrategy.COSINE
    )
    
    # Retriever-Objekt erstellen
    # search_kwargs kann k (Anzahl Dokumente) und andere Parameter enthalten
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4}  # Standardmäßig 4 relevanteste Dokumente abrufen
    )
    
    print(f"✅ VectorStore erfolgreich geladen!")
    print(f"📊 Anzahl gespeicherter Embeddings: {vectorstore.index.ntotal}")
    
    return retriever, vectorstore

print("✅ Retriever-Funktion definiert!")

In [ ]:
# Retriever aus gespeicherter Datenbank laden
react_retriever, react_vectorstore_loaded = load_retriever_from_vectordb(
    "../vector_databases/vector_db_react"
)

print("\n" + "="*80)
print("📝 RETRIEVER-TEST:")
print("="*80)

In [ ]:
# Testen Sie Ihr Retrieval-System mit Anfragen
test_query = "What is ReAct?"

print(f"🔍 Anfrage: '{test_query}'\n")

# Relevante Dokumente abrufen
relevant_docs = react_retriever.get_relevant_documents(test_query)

print(f"📚 Gefundene Dokumente: {len(relevant_docs)}\n")

# Zeige die ersten 2 gefundenen Dokumente
for i, doc in enumerate(relevant_docs[:2]):
    print(f"--- DOKUMENT {i+1} ---")
    print(f"Text: {doc.page_content[:200]}...")
    print(f"Metadaten: {doc.metadata}")
    print()

In [ ]:
# Weitere Testanfrage auf Deutsch
test_query_de = "Was ist die Hauptidee von ReAct?"

print(f"🔍 Anfrage (Deutsch): '{test_query_de}'\n")

relevant_docs_de = react_retriever.get_relevant_documents(test_query_de)

print(f"📚 Gefundene Dokumente: {len(relevant_docs_de)}\n")

for i, doc in enumerate(relevant_docs_de[:1]):
    print(f"--- DOKUMENT {i+1} ---")
    print(f"Text: {doc.page_content}")
    print()

---

## **Übung 6: Verbindung von Retrieval mit LLM**

Sie verbinden nun das Dokumenten-Retrieval mit dem Language Model.

**Schritte:**

1. **Retrieval Chain erstellen:** Verknüpfen Sie Ihr Retrieval-System mit Ihrem LLM
2. **Chain testen:** Bestätigen Sie die Funktionsweise durch Generieren von Antworten aus abgerufenen Dokumenten

### **Was passiert hier?**
1. **create_stuff_documents_chain:** Nimmt abgerufene Dokumente und fügt sie in einen Prompt ein
2. **create_retrieval_chain:** Verbindet Retriever → Document Chain → LLM
3. **Workflow:**
   - Benutzer stellt Frage
   - Retriever findet relevante Dokumente
   - Dokumente werden in Prompt eingefügt
   - LLM generiert Antwort basierend auf Dokumenten

### **Vorteile:**
- LLM hat Zugriff auf aktuelle, spezifische Informationen
- Reduziert Halluzinationen (erfundene Fakten)
- Antworten sind nachvollziehbar (durch Quelldokumente)

In [ ]:
# Zusätzliche Imports für Chains
from langchain import hub
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

# Funktion zum Erstellen von Retrieval- und Dokumentenverarbeitungs-Chains
def create_rag_chain(retriever, llm):
    """
    Erstellt eine vollständige RAG-Chain (Retrieval-Augmented Generation).
    
    Args:
        retriever: Retriever-Objekt zum Abrufen relevanter Dokumente
        llm: Language Model zum Generieren von Antworten
    
    Returns:
        RetrievalChain: Vollständige RAG-Chain
    """
    print("⏳ Erstelle RAG-Chain...")
    
    # Schritt 1: Prompt-Template von LangChain Hub laden
    # Dieses Template formatiert Dokumente und Fragen für das LLM
    prompt = hub.pull("langchain-ai/retrieval-qa-chat")
    
    # Schritt 2: Document Chain erstellen
    # Diese Chain nimmt Dokumente und erstellt einen Prompt daraus
    stuff_documents_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt
    )
    
    # Schritt 3: Retrieval Chain erstellen
    # Diese Chain kombiniert Retriever und Document Chain
    retrieval_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=stuff_documents_chain
    )
    
    print("✅ RAG-Chain erfolgreich erstellt!")
    print("🔗 Chain-Struktur: Anfrage → Retriever → Dokumente → LLM → Antwort")
    
    return retrieval_chain

print("✅ RAG-Chain-Funktion definiert!")

In [ ]:
# RAG-Chain mit Retriever und LLM erstellen
react_rag_chain = create_rag_chain(retriever=react_retriever, llm=llm)

In [ ]:
# Chain mit einer Beispielfrage aufrufen
question = "What is ReAct and why is it important?"

print(f"❓ Frage: {question}\n")
print("⏳ Generiere Antwort...\n")

# Chain aufrufen
result = react_rag_chain.invoke({"input": question})

print("="*80)
print("🤖 ANTWORT:")
print("="*80)
print(result['answer'])

print("\n" + "="*80)
print("📚 VERWENDETE QUELLEN:")
print("="*80)
for i, doc in enumerate(result['context'][:2]):
    print(f"\nQuelle {i+1}:")
    print(doc.page_content[:200] + "...")

---

## **Übung 7: Interaktives Chat-System**

In der letzten Übung bauen Sie ein interaktives chatbasiertes Abfragesystem.

**Schritte:**

1. **Chat-Schnittstelle erstellen:** Entwickeln Sie eine einfache Funktion für interaktive Abfragen
2. **Chat ausführen:** Ermöglichen Sie Benutzern, Fragen zu stellen und sofortige Antworten zu erhalten

### **Praktisches Beispiel:**
Wir verwenden das Paracetamol-PDF, um ein medizinisches Informationssystem zu demonstrieren.

In [ ]:
# Laden Sie das Paracetamol-PDF
print("📄 Lade Paracetamol-Dokument...")
paracetamol_docs = load_pdf(pdf_path="../documents/paracetamol.pdf")

In [ ]:
# Dokumente in Chunks aufteilen
print("\n📦 Teile Dokument in Chunks...")
paracetamol_chunks = chunk_documents(paracetamol_docs, chunk_size=300, chunk_overlap=50)

In [ ]:
# Embeddings erstellen und speichern
print("\n💾 Erstelle und speichere Embeddings...")
paracetamol_vectorstore = embed_and_store(chunks=paracetamol_chunks, db_name="paracetamol")

In [ ]:
# Retriever erstellen
print("\n🔍 Erstelle Retriever...")
paracetamol_retriever, _ = load_retriever_from_vectordb(
    "../vector_databases/vector_db_paracetamol"
)

In [ ]:
# RAG-Chain erstellen
print("\n🔗 Erstelle RAG-Chain...")
paracetamol_rag_chain = create_rag_chain(retriever=paracetamol_retriever, llm=llm)

In [ ]:
# Definieren Sie Ihre interaktive Chat-Abfragefunktion
def chat_with_pdf(question, rag_chain=paracetamol_rag_chain, verbose=True):
    """
    Interaktive Chat-Funktion für Fragen zum PDF.
    
    Args:
        question (str): Die zu stellende Frage
        rag_chain: Die zu verwendende RAG-Chain
        verbose (bool): Ob zusätzliche Informationen angezeigt werden sollen
    
    Returns:
        str: Die generierte Antwort
    """
    if verbose:
        print("="*80)
        print(f"❓ FRAGE: {question}")
        print("="*80)
        print("⏳ Suche relevante Informationen...\n")
    
    # RAG-Chain aufrufen
    result = rag_chain.invoke({"input": question})
    
    if verbose:
        print("🤖 ANTWORT:")
        print("-" * 80)
    
    print(result['answer'])
    
    if verbose:
        print("\n" + "="*80)
        print(f"📊 Anzahl verwendeter Quellen: {len(result['context'])}")
        print("="*80)
    
    return result['answer']

print("✅ Chat-Funktion definiert!")

In [ ]:
# Testen Sie Ihr interaktives Chat-System

# Frage 1: Zusammenfassung auf Englisch
chat_with_pdf("Give me a summary of Paracetamol in 3 sentences.")

In [ ]:
# Frage 2: Zusammenfassung auf Deutsch
print("\n\n")
chat_with_pdf("Gib mir eine Zusammenfassung von Paracetamol in 3 Sätzen.")

In [ ]:
# Frage 3: Spezifische medizinische Frage
print("\n\n")
chat_with_pdf("What are the main side effects of Paracetamol?")

In [ ]:
# Frage 4: Dosierungsfrage
print("\n\n")
chat_with_pdf("What is the recommended dosage of Paracetamol?")

---

## **Fazit & Reflexion**

### Was haben Sie gelernt?

1. **RAG-Grundlagen:**
   - Retrieval-Augmented Generation kombiniert Retrieval und Generierung
   - Reduziert Halluzinationen durch Verwendung echter Dokumente
   - Ermöglicht aktuelle und domänenspezifische Informationen

2. **Technische Umsetzung:**
   - PDF-Laden mit PyPDFLoader
   - Chunking mit RecursiveCharacterTextSplitter
   - Embeddings mit HuggingFace sentence-transformers
   - Vektorspeicherung mit FAISS
   - Chain-Erstellung mit LangChain

3. **Wichtige Konzepte:**
   - Chunk Size und Overlap balancieren
   - Embedding-Normalisierung für korrekte Cosine Similarity
   - Gleiches Embedding-Modell für Speichern und Laden verwenden
   - k-Parameter für Anzahl abgerufener Dokumente

### Limitationen des Systems:

1. **Qualität abhängig von Quelldokumenten:**
   - Kann nur Informationen aus bereitgestellten PDFs nutzen
   - Falsche oder veraltete Dokumente → falsche Antworten

2. **Chunk-Größe ist kritisch:**
   - Zu klein: Kontext geht verloren
   - Zu groß: Irrelevante Informationen verwässern Antwort

3. **Retrieval-Qualität:**
   - Semantische Suche ist nicht perfekt
   - Manchmal werden nicht die relevantesten Chunks gefunden

4. **LLM-Limitationen:**
   - Auch mit RAG können LLMs Fehler machen
   - Kontext-Fenster begrenzt Anzahl verwendbarer Dokumente

### Verbesserungsmöglichkeiten:

1. **Hybride Suche:**
   - Kombination von Keyword-Suche und semantischer Suche
   - BM25 + FAISS für bessere Ergebnisse

2. **Reranking:**
   - Zweite Stufe zum Neusortieren abgerufener Dokumente
   - Verbessert Relevanz der verwendeten Chunks

3. **Metadaten-Filterung:**
   - Nach Datum, Autor, Dokumenttyp filtern
   - Präzisere Suche in großen Datenbanken

4. **Chat-Historie:**
   - Konversationsgedächtnis für Follow-up-Fragen
   - Bessere Benutzererfahrung

5. **Größere Modelle:**
   - Llama 3.1 70B oder GPT-4 für bessere Antworten
   - Längere Kontextfenster für mehr Dokumente

### Praktische Anwendungen:

- **Medizin:** Abfrage von medizinischen Leitlinien und Fachliteratur
- **Recht:** Suche in Gesetzen und Urteilen
- **Kundenservice:** FAQ-Bots mit Produktdokumentation
- **Forschung:** Literaturrecherche und Paper-Zusammenfassungen
- **Unternehmen:** Interne Wissensdatenbanken durchsuchbar machen

---

## Herzlichen Glückwunsch! 🎉

Sie haben erfolgreich ein vollständiges RAG-System von Grund auf erstellt!

**Nächste Schritte:**
- Experimentieren Sie mit verschiedenen PDF-Dokumenten
- Passen Sie Chunk-Größen an
- Testen Sie verschiedene Embedding-Modelle
- Implementieren Sie eigene Verbesserungen

**Weiterführende Ressourcen:**
- [LangChain Dokumentation](https://python.langchain.com/)
- [FAISS GitHub](https://github.com/facebookresearch/faiss)
- [Hugging Face Models](https://huggingface.co/models)
- [RAG Papers](https://arxiv.org/abs/2005.11401)